# Supply Chain Planning

**The problem.** A company makes 2 products and sells them in 9 markets. Goods move through a simple network:

- A **supplier** provides a limited amount of raw material (RM) for each product.
- **3 factories** turn the raw material into finished products.
- Products are shipped either **directly to a market** or **through one of 5 warehouses**. Factory-to-warehouse shipping has 2 transport modes.

**The decision.** How much of each product to ship along each route. This also sets how much each factory produces (everything it ships out) and how much demand goes unmet.

**The goal.** Maximise profit:

> sales revenue − raw material cost − production cost − transport cost − penalty for unmet demand

**The rules (constraints).**

1. **Raw material:** total production of each product, across all factories, cannot exceed the supplier's limit.
2. **Production capacity:** each factory has 2 operations shared by both products; its product mix must fit within each operation's capacity.
3. **Demand:** each market's demand for each product is met by shipments, and any gap is recorded as a shortage.
4. **Warehouse balance:** everything that goes into a warehouse must come out of it.
5. **Unavailable routes:** routes with a transport cost of 100 don't exist, so no flow is allowed on them.

All decisions are continuous, so this is a **linear program (LP)**. The notebook builds it, solves it with Gurobi, and reports the optimal plan, its sensitivity analysis (shadow prices, slacks, reduced costs), and the production at each factory.

In [2]:
# install the third-party packages into this notebook's active Python environment
# gurobipy ships with a size-limited license, which is enough for this model
%pip install --quiet --disable-pip-version-check "gurobipy>=11" "pandas>=1.5"

In [3]:
# import the Gurobi Python API and its constants (GRB.MAXIMIZE, GRB.OPTIMAL, ...)
import gurobipy as gp
from gurobipy import GRB

In [4]:
# create the model; all decision variables are continuous,
# so Gurobi solves it as a linear program (LP)
SC = gp.Model("Supply_Chain_Planning")

Restricted license - for non-production use only - expires 2027-11-29


In [5]:
# products
P = [1,2]
# operations
O = [1,2]
# factories
F = [1,2,3]
# warehouses
W = [1,2,3,4,5]
# markets
M = [1,2,3,4,5,6,7,8,9]
# transport modes (factory to warehouse only)
MD = [1,2]

In [6]:
# supplier RM capacity (product)
S = {1: 12, 2: 10.5}

In [7]:
# shared production capacity of each operation (operation, product)
CAP = {(1,1): 5, (1,2): 5, (2,1): 8, (2,2): 4}

In [8]:
# RM cost (product)
RC = {1: 4, 2: 3}

In [9]:
# variable production cost (factory, product)
PC = {(1,1): 0.15, (1,2): 0.1, (2,1): 0.19, (2,2): 0.14, (3,1): 0.13, (3,2): 0.09}

In [10]:
# transport cost - factory to market (factory, market)
TCFM = {(1,1): 100, (1,2): 0.6, (1,3): 100, (1,4): 100, (1,5): 100, (1,6): 100, (1,7): 100, (1,8): 100, (1,9): 100,
        (2,1): 100, (2,2): 100, (2,3): 100, (2,4): 100, (2,5): 100, (2,6): 100, (2,7): 100, (2,8): 100, (2,9): 100,
        (3,1): 100, (3,2): 100, (3,3): 100, (3,4): 100, (3,5): 100, (3,6): 100, (3,7): 0.5, (3,8): 0.6, (3,9): 100}

In [11]:
# transport cost - factory to warehouse (factory, warehouse, transport mode)
TCFW = {(1,1,1): 1.1, (1,1,2): 100, (1,2,1): 0.7, (1,2,2): 0.5, (1,3,1): 0.6, (1,3,2): 100, (1,4,1): 0.7, (1,4,2): 100, (1,5,1): 0.9, (1,5,2): 0.7, 
       (2,1,1): 0.6, (2,1,2): 100, (2,2,1): 0.9, (2,2,2): 0.7, (2,3,1): 1.0, (2,3,2): 0.7, (2,4,1): 0.5, (2,4,2): 100, (2,5,1): 100, (2,5,2): 100,
       (3,1,1): 100, (3,1,2): 100, (3,2,1): 0.8, (3,2,2): 100, (3,3,1): 0.9, (3,3,2): 100, (3,4,1): 0.7, (3,4,2): 100, (3,5,1): 0.6, (3,5,2): 100}

In [12]:
# transport cost - warehouse to market (warehouse, market)
TCWM = {(1,1): 0.1, (1,2): 0.2, (1,3): 0.15, (1,4): 0.12, (1,5): 0.18, (1,6): 0.1, (1,7): 100, (1,8): 100, (1,9): 100,
       (2,1): 0.15, (2,2): 0.12, (2,3): 0.1, (2,4): 0.17, (2,5): 100, (2,6): 100, (2,7): 100, (2,8): 100, (2,9): 100,
       (3,1): 0.18, (3,2): 0.15, (3,3): 100, (3,4): 0.15, (3,5): 100, (3,6): 100, (3,7): 100, (3,8): 100, (3,9): 100,
       (4,1): 100, (4,2): 100, (4,3): 100, (4,4): 0.14, (4,5): 0.13, (4,6): 100, (4,7): 0.18, (4,8): 0.1, (4,9): 0.14,
       (5,1): 100, (5,2): 100, (5,3): 100, (5,4): 100, (5,5): 100, (5,6): 0.19, (5,7): 0.14, (5,8): 0.16, (5,9): 0.12}

In [13]:
# demand (market, product)
DEM = {(1,1): 0.750, (1,2): 1.000, (2,1): 1.300, (2,2): 0, (3,1): 0.850, (3,2): 1.450, (4,1): 0.750, (4,2): 1.250,
      (5,1): 0.550, (5,2): 0.950, (6,1): 0.200, (6,2): 0.600, (7,1): 0.700, (7,2): 1.500, (8,1): 0.450, (8,2): 1.500,
      (9,1): 0.750, (9,2): 0.900}

In [14]:
# sales price (market, product)
SP = {(1,1): 6.5, (1,2): 5.5, (2,1): 7, (2,2): 5.1, (3,1): 6.3, (3,2): 5, (4,1): 7.6, (4,2): 5,
      (5,1): 7.2, (5,2): 5.4, (6,1): 6.5, (6,2): 5.2, (7,1): 7.9, (7,2): 5.7, (8,1): 7.2, (8,2): 5,
      (9,1): 7, (9,2): 5.2}

In [15]:
# penalty cost per unit of unmet demand
COST_SHORT = 100
# transport cost that marks a lane as unavailable; flow on such lanes is fixed to 0
NO_LANE = 100

In [16]:
# define variables - factory to market flows (product, factory, market)
FM = {(p,f,m): SC.addVar(lb=0.0, ub=0.0 if TCFM[f,m] >= NO_LANE else GRB.INFINITY, name=f"FM_{p}_{f}_{m}") for p in P for f in F for m in M}
SC.update()
FM

In [17]:
# define variables - factory to warehouse flows (product, factory, warehouse, transport mode)
FW = {(p,f,w,md): SC.addVar(lb=0.0, ub=0.0 if TCFW[f,w,md] >= NO_LANE else GRB.INFINITY, name=f"FW_{p}_{f}_{w}_{md}") for p in P for f in F for w in W for md in MD}
SC.update()
FW

In [18]:
# define variables - warehouse to market flows (product, warehouse, market)
WM = {(p,w,m): SC.addVar(lb=0.0, ub=0.0 if TCWM[w,m] >= NO_LANE else GRB.INFINITY, name=f"WM_{p}_{w}_{m}") for p in P for w in W for m in M}
SC.update()
WM

In [19]:
# define variables - shortage (unmet demand) (market, product)
SHORT = {(m,p): SC.addVar(lb=0.0, ub=GRB.INFINITY, name=f"SHORT_{m}_{p}") for m in M for p in P}

In [20]:
# add constraint: supplier RM limit
for p in P:
    SC.addConstr(gp.quicksum(FW[p,f,w,md] for f in F for w in W for md in MD) + gp.quicksum(FM[p,f,m] for f in F for m in M) <= S[p], name=f"RM_supplier_limit_{p}")

In [21]:
# add constraint: production capacity
# CAP[o,p] is how much of product p operation o could make if it made only p.
# at each factory, the share of each operation's capacity used by all products must not exceed 100%
for f in F:
    for o in O:
        SC.addConstr(gp.quicksum((gp.quicksum(FW[p,f,w,md] for w in W for md in MD) + gp.quicksum(FM[p,f,m] for m in M))/CAP[o,p] for p in P) <= 1, name=f"production_capacity_{f}_{o}")

In [22]:
# add constraint: demand is met by shipments, and any gap is recorded as shortage
for m in M:
    for p in P:
        SC.addConstr(gp.quicksum(FM[p,f,m] for f in F) + gp.quicksum(WM[p,w,m] for w in W) + SHORT[m,p] == DEM[m,p], name=f"demand_{m}_{p}")

In [23]:
# add constraint: warehouse flow balance (inflow = outflow)
for w in W:
    for p in P:
        SC.addConstr(gp.quicksum(FW[p,f,w,md] for f in F for md in MD) == gp.quicksum(WM[p,w,m] for m in M), name=f"flow_balance_{w}_{p}")

In [24]:
# add the objective function:
# profit = sales revenue - RM cost - production cost - transport cost (factory-market, factory-warehouse, warehouse-market) - shortage penalty
SC.setObjective(gp.quicksum(SP[m,p]*(gp.quicksum(FM[p,f,m] for f in F) + gp.quicksum(WM[p,w,m] for w in W)) for m in M for p in P) - gp.quicksum(RC[p]*(gp.quicksum(FW[p,f,w,md] for f in F for w in W for md in MD) + gp.quicksum(FM[p,f,m] for f in F for m in M)) for p in P) - gp.quicksum(PC[f,p]*(gp.quicksum(FW[p,f,w,md] for w in W for md in MD) + gp.quicksum(FM[p,f,m] for m in M)) for f in F for p in P) - gp.quicksum(TCFM[f,m]*FM[p,f,m] for p in P for f in F for m in M) - gp.quicksum(TCFW[f,w,md]*FW[p,f,w,md] for p in P for f in F for w in W for md in MD) - gp.quicksum(TCWM[w,m]*WM[p,w,m] for p in P for w in W for m in M) - gp.quicksum(COST_SHORT*SHORT[m,p] for m in M for p in P), GRB.MAXIMIZE)

In [25]:
# solve the model and stop if no optimal solution was found
SC.optimize()
if SC.Status != GRB.OPTIMAL:
    raise RuntimeError(f"The solver did not find an optimal solution (status code: {SC.Status}).")
print("Solver status: OPTIMAL")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13700HX, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 24 logical processors, using up to 24 threads

Optimize a model with 36 rows, 222 columns and 654 nonzeros (Max)
Model fingerprint: 0xe9e5951a
Model has 222 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-01, 1e+00]
  Objective range  [1e+00, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e-01, 1e+01]

Presolve removed 1 rows and 133 columns
Presolve time: 0.00s
Presolved: 35 rows, 89 columns, 223 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    9.0513000e+01   1.075000e+01   0.000000e+00      0s
      33   -1.7432000e+01   0.000000e+00   0.000000e+00      0s

Solved in 33 iterations and 0.00 seconds (0.00 work units)
Optimal objective -1.743200000e+01
Solver status: OPTIMAL


In [26]:
# print every variable's value and the optimal profit
for v in SC.getVars():
    print(v.VarName, v.X)
print("Objective Fn Value =", SC.ObjVal)

FM_1_1_1 0.0
FM_1_1_2 1.3
FM_1_1_3 0.0
FM_1_1_4 0.0
FM_1_1_5 0.0
FM_1_1_6 0.0
FM_1_1_7 0.0
FM_1_1_8 0.0
FM_1_1_9 0.0
FM_1_2_1 0.0
FM_1_2_2 0.0
FM_1_2_3 0.0
FM_1_2_4 0.0
FM_1_2_5 0.0
FM_1_2_6 0.0
FM_1_2_7 0.0
FM_1_2_8 0.0
FM_1_2_9 0.0
FM_1_3_1 0.0
FM_1_3_2 0.0
FM_1_3_3 0.0
FM_1_3_4 0.0
FM_1_3_5 0.0
FM_1_3_6 0.0
FM_1_3_7 0.7
FM_1_3_8 0.45
FM_1_3_9 0.0
FM_2_1_1 0.0
FM_2_1_2 0.0
FM_2_1_3 0.0
FM_2_1_4 0.0
FM_2_1_5 0.0
FM_2_1_6 0.0
FM_2_1_7 0.0
FM_2_1_8 0.0
FM_2_1_9 0.0
FM_2_2_1 0.0
FM_2_2_2 0.0
FM_2_2_3 0.0
FM_2_2_4 0.0
FM_2_2_5 0.0
FM_2_2_6 0.0
FM_2_2_7 0.0
FM_2_2_8 0.0
FM_2_2_9 0.0
FM_2_3_1 0.0
FM_2_3_2 0.0
FM_2_3_3 0.0
FM_2_3_4 0.0
FM_2_3_5 0.0
FM_2_3_6 0.0
FM_2_3_7 1.5
FM_2_3_8 1.5
FM_2_3_9 0.0
FW_1_1_1_1 0.0
FW_1_1_1_2 0.0
FW_1_1_2_1 0.0
FW_1_1_2_2 1.0000000000000009
FW_1_1_3_1 0.0
FW_1_1_3_2 0.0
FW_1_1_4_1 0.0
FW_1_1_4_2 0.0
FW_1_1_5_1 0.0
FW_1_1_5_2 0.0
FW_1_2_1_1 0.6999999999999995
FW_1_2_1_2 0.0
FW_1_2_2_1 0.0
FW_1_2_2_2 0.0
FW_1_2_3_1 0.0
FW_1_2_3_2 0.0
FW_1_2_4_1 1.3
FW_1_2_4_2 0

In [27]:
# constraint report: shadow price = change in profit per extra unit of the constraint's right-hand side;
# slack = unused room in the constraint (0 means the constraint is binding)
import pandas as pd
result_constraints = [{'name': c.ConstrName, 'shadow price': c.Pi, 'slack': c.Slack}
     for c in SC.getConstrs()]
print(pd.DataFrame(result_constraints))

                       name  shadow price   slack
0       RM_supplier_limit_1          0.00  5.7000
1       RM_supplier_limit_2          0.00  1.8000
2   production_capacity_1_1        506.15  0.0000
3   production_capacity_1_2          0.00  0.0375
4   production_capacity_2_1        505.30  0.0000
5   production_capacity_2_2          0.64  0.0000
6   production_capacity_3_1        505.15  0.0000
7   production_capacity_3_2          0.64  0.0000
8                demand_1_1        -99.53  0.0000
9                demand_1_2        -99.48  0.0000
10               demand_2_1        -98.98  0.0000
11               demand_2_2          0.00  0.0000
12               demand_3_1        -99.68  0.0000
13               demand_3_2        -99.93  0.0000
14               demand_4_1        -98.37  0.0000
15               demand_4_2       -100.00  0.0000
16               demand_5_1        -98.76  0.0000
17               demand_5_2        -99.59  0.0000
18               demand_6_1        -99.53  0.0000


In [28]:
# variables used in the optimal plan (value >= 0.001), with their reduced costs
result_variables = [{'name': v.VarName, 'value': v.X, 'reduced cost': v.RC}
     for v in SC.getVars() if v.X >= 0.001]
print(pd.DataFrame(result_variables))

          name  value  reduced cost
0     FM_1_1_2   1.30           0.0
1     FM_1_3_7   0.70           0.0
2     FM_1_3_8   0.45           0.0
3     FM_2_3_7   1.50           0.0
4     FM_2_3_8   1.50           0.0
5   FW_1_1_2_2   1.00           0.0
6   FW_1_2_1_1   0.70           0.0
7   FW_1_2_4_1   1.30           0.0
8   FW_1_3_5_1   0.85           0.0
9   FW_2_1_2_2   2.70           0.0
10  FW_2_2_1_1   0.60           0.0
11  FW_2_2_4_1   2.40           0.0
12    WM_1_1_1   0.60           0.0
13    WM_1_1_6   0.10           0.0
14    WM_1_2_1   0.15           0.0
15    WM_1_2_3   0.85           0.0
16    WM_1_4_4   0.75           0.0
17    WM_1_4_5   0.55           0.0
18    WM_1_5_6   0.10           0.0
19    WM_1_5_9   0.75           0.0
20    WM_2_1_6   0.60           0.0
21    WM_2_2_1   1.00           0.0
22    WM_2_2_3   1.45           0.0
23    WM_2_2_4   0.25           0.0
24    WM_2_4_4   0.55           0.0
25    WM_2_4_5   0.95           0.0
26    WM_2_4_9   0.90       

In [29]:
# variables not used in the optimal plan (value < 0.001); reduced cost = how much profit
# would change per unit if the variable were forced into the plan
result_variables = [{'name': v.VarName, 'value': v.X, 'reduced cost': v.RC}
     for v in SC.getVars() if v.X < 0.001]
print(pd.DataFrame(result_variables))

          name  value  reduced cost
0     FM_1_1_1    0.0        -99.35
1     FM_1_1_3    0.0        -99.40
2     FM_1_1_4    0.0        -99.41
3     FM_1_1_5    0.0        -99.42
4     FM_1_1_6    0.0        -99.35
..         ...    ...           ...
189  SHORT_7_2    0.0         -0.92
190  SHORT_8_1    0.0         -1.36
191  SHORT_8_2    0.0         -0.12
192  SHORT_9_1    0.0         -1.04
193  SHORT_9_2    0.0         -0.20

[194 rows x 3 columns]


In [30]:
# total production of each product at each factory (direct shipments + shipments to warehouses)
for f in F:
    for p in P:
        X = sum(FM[p,f,m].X for m in M) + sum(FW[p,f,w,md].X for w in W for md in MD)
        print("Production at factory %g, product %g : " % (f,p), X)

Production at factory 1, product 1 :  2.3000000000000007
Production at factory 1, product 2 :  2.699999999999999
Production at factory 2, product 1 :  1.9999999999999996
Production at factory 2, product 2 :  3.0000000000000004
Production at factory 3, product 1 :  1.9999999999999996
Production at factory 3, product 2 :  3.0


In [31]:
# ==============================================================================
# MANAGERIAL DASHBOARD: SUPPLY CHAIN OPTIMIZATION COCKPIT
# ==============================================================================
# Generates and displays an executive HTML dashboard directly inside this notebook,
# bridging revenue, logistics costs, factory bottlenecks, and market service levels.

from IPython.display import display, HTML
import json

# 1. Financial & Operational Aggregations
gross_revenue = sum(SP[m,p] * (sum(FM[p,f,m].X for f in F) + sum(WM[p,w,m].X for w in W)) for m in M for p in P)
rm_cost = sum(RC[p] * (sum(FW[p,f,w,md].X for f in F for w in W for md in MD) + sum(FM[p,f,m].X for f in F for m in M)) for p in P)
prod_cost = sum(PC[f,p] * (sum(FW[p,f,w,md].X for w in W for md in MD) + sum(FM[p,f,m].X for m in M)) for f in F for p in P)
trans_cost = (
    sum(TCFM[f,m] * FM[p,f,m].X for p in P for f in F for m in M if TCFM[f,m] < NO_LANE) +
    sum(TCFW[f,w,md] * FW[p,f,w,md].X for p in P for f in F for w in W for md in MD if TCFW[f,w,md] < NO_LANE) +
    sum(TCWM[w,m] * WM[p,w,m].X for p in P for w in W for m in M if TCWM[w,m] < NO_LANE)
)
shortage_penalty = sum(COST_SHORT * SHORT[m,p].X for m in M for p in P)
op_profit = gross_revenue - rm_cost - prod_cost - trans_cost
net_obj = SC.ObjVal

total_demand = sum(DEM.values())
shortage_vol = sum(SHORT[m,p].X for m in M for p in P)
fulfilled_vol = total_demand - shortage_vol
service_level = (fulfilled_vol / total_demand) * 100

# 2. Factory Utilization
fact_data = {}
for f in F:
    p1 = sum(FM[1,f,m].X for m in M) + sum(FW[1,f,w,md].X for w in W for md in MD)
    p2 = sum(FM[2,f,m].X for m in M) + sum(FW[2,f,w,md].X for w in W for md in MD)
    tot = p1 + p2
    u1 = (p1 / CAP[1,1] + p2 / CAP[1,2]) * 100
    u2 = (p1 / CAP[2,1] + p2 / CAP[2,2]) * 100
    fact_data[f] = {'p1': p1, 'p2': p2, 'tot': tot, 'u1': u1, 'u2': u2}

# 3. Market Summary Rows HTML
mkt_rows = ""
for m in M:
    d1 = DEM[m,1]; s1 = sum(FM[1,f,m].X for f in F) + sum(WM[1,w,m].X for w in W)
    d2 = DEM[m,2]; s2 = sum(FM[2,f,m].X for f in F) + sum(WM[2,w,m].X for w in W)
    tot_d = d1 + d2; tot_s = s1 + s2
    sh = SHORT[m,1].X + SHORT[m,2].X
    rev = s1 * SP[m,1] + s2 * SP[m,2]
    fill = (tot_s / tot_d * 100) if tot_d > 0 else 100
    is_short = sh > 0.001
    bg = 'background-color: #fff1f2;' if is_short else ''
    badge = f"<span style='background:#fee2e2;color:#b91c1c;padding:2px 6px;border-radius:999px;font-size:11px;font-weight:700;'>{fill:.1f}%</span>" if is_short else f"<span style='background:#dcfce7;color:#15803d;padding:2px 6px;border-radius:999px;font-size:11px;font-weight:700;'>{fill:.1f}%</span>"
    ch = "Direct (F1)" if m == 2 else ("Direct (F3)" if m in [7,8] else "Via Warehouse")
    mkt_rows += f"""<tr style='border-bottom:1px solid #f1f5f9;{bg}'>
      <td style='padding:8px 12px;font-weight:bold;color:#0f172a;'>Market {m}</td>
      <td style='padding:8px 12px;text-align:center;'>{d1:.2f} / <b style='color:#2563eb;'>{s1:.2f}</b></td>
      <td style='padding:8px 12px;text-align:center;'>{d2:.2f} / <b style='color:#d97706;'>{s2:.2f}</b></td>
      <td style='padding:8px 12px;text-align:center;font-weight:bold;'>{tot_d:.2f}</td>
      <td style='padding:8px 12px;text-align:center;color:{'#e11d48' if is_short else '#94a3b8'};font-weight:bold;'>{f'{sh:.2f}' if is_short else '-'}</td>
      <td style='padding:8px 12px;text-align:center;'>{badge}</td>
      <td style='padding:8px 12px;text-align:right;font-weight:bold;color:#0f172a;'>${rev:.2f}k</td>
      <td style='padding:8px 12px;text-align:center;color:#64748b;font-size:11px;'>{ch}</td>
    </tr>"""

# 4. Construct Full Self-Contained Dashboard HTML
dashboard_html = f"""
<div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;background:#f8fafc;color:#0f172a;padding:20px;border-radius:14px;border:1px solid #e2e8f0;margin:15px 0;">
  
  <!-- Header Banner -->
  <div style="background:#0f172a;color:#ffffff;padding:16px 20px;border-radius:10px;display:flex;justify-content:space-between;align-items:center;margin-bottom:18px;">
    <div>
      <div style="display:flex;align-items:center;gap:10px;">
        <span style="background:#2563eb;color:#fff;font-weight:900;padding:4px 8px;border-radius:6px;font-size:14px;">SC</span>
        <h2 style="margin:0;font-size:18px;font-weight:700;color:#ffffff;">Supply Chain Optimization Managerial Cockpit</h2>
        <span style="background:#064e3b;color:#34d399;border:1px solid #059669;padding:2px 8px;border-radius:999px;font-size:11px;font-weight:700;">GUROBI LP OPTIMAL</span>
      </div>
      <p style="margin:4px 0 0 0;font-size:12px;color:#94a3b8;">End-to-end Prescriptive Linear Programming Analysis &bull; 2 Products &bull; 3 Factories &bull; 5 Warehouses &bull; 9 Markets</p>
    </div>
  </div>

  <!-- Strategic Alert -->
  <div style="background:#fffbeb;border-left:4px solid #f59e0b;padding:12px 16px;border-radius:0 8px 8px 0;margin-bottom:18px;display:flex;justify-content:space-between;align-items:center;">
    <div>
      <div style="font-weight:700;font-size:12px;color:#b45309;text-transform:uppercase;">🚨 Key Finding: 100% Factory Bottleneck Constraint</div>
      <div style="font-size:12px;color:#78350f;margin-top:2px;">
        All 3 factories operate at <b>100% maximum capacity</b> on Operation 1 (15.00u output vs 15.45u demand).
        This forces a <b>0.45 unit shortage in Market 4 (Product 2)</b>, triggering a <b>-$45.00k SLA penalty</b> that drops net objective to <b>-${abs(net_obj):.2f}k</b> despite healthy physical <b>+${op_profit:.2f}k operating profit</b> (30.6% margin).
      </div>
    </div>
  </div>

  <!-- KPI Scorecard -->
  <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(130px, 1fr));gap:10px;margin-bottom:20px;">
    <div style="background:#fff;border:1px solid #e2e8f0;padding:12px;border-radius:8px;">
      <div style="font-size:10px;color:#64748b;font-weight:700;text-transform:uppercase;">Gross Revenue</div>
      <div style="font-size:19px;font-weight:800;color:#0f172a;margin-top:2px;">${gross_revenue:.2f}k</div>
      <div style="font-size:11px;color:#16a34a;margin-top:2px;">15.00 u delivered</div>
    </div>
    <div style="background:#fff;border:1px solid #e2e8f0;padding:12px;border-radius:8px;">
      <div style="font-size:10px;color:#64748b;font-weight:700;text-transform:uppercase;">Raw Materials</div>
      <div style="font-size:19px;font-weight:800;color:#0f172a;margin-top:2px;">${rm_cost:.2f}k</div>
      <div style="font-size:11px;color:#64748b;margin-top:2px;">82.1% of cost</div>
    </div>
    <div style="background:#fff;border:1px solid #e2e8f0;padding:12px;border-radius:8px;">
      <div style="font-size:10px;color:#64748b;font-weight:700;text-transform:uppercase;">Mfg Processing</div>
      <div style="font-size:19px;font-weight:800;color:#0f172a;margin-top:2px;">${prod_cost:.2f}k</div>
      <div style="font-size:11px;color:#64748b;margin-top:2px;">F1, F2, F3</div>
    </div>
    <div style="background:#fff;border:1px solid #e2e8f0;padding:12px;border-radius:8px;">
      <div style="font-size:10px;color:#64748b;font-weight:700;text-transform:uppercase;">Freight Cost</div>
      <div style="font-size:19px;font-weight:800;color:#0f172a;margin-top:2px;">${trans_cost:.2f}k</div>
      <div style="font-size:11px;color:#64748b;margin-top:2px;">Road & Rail lanes</div>
    </div>
    <div style="background:#f0fdf4;border:1px solid #bbf7d0;padding:12px;border-radius:8px;">
      <div style="font-size:10px;color:#166534;font-weight:700;text-transform:uppercase;">Operating Profit</div>
      <div style="font-size:19px;font-weight:800;color:#15803d;margin-top:2px;">+${op_profit:.2f}k</div>
      <div style="font-size:11px;color:#166534;margin-top:2px;">30.6% Op Margin</div>
    </div>
    <div style="background:#fff1f2;border:1px solid #fecdd3;padding:12px;border-radius:8px;">
      <div style="font-size:10px;color:#9f1239;font-weight:700;text-transform:uppercase;">Shortage Penalty</div>
      <div style="font-size:19px;font-weight:800;color:#be123c;margin-top:2px;">-${shortage_penalty:.2f}k</div>
      <div style="font-size:11px;color:#be123c;margin-top:2px;">0.45 u in M4</div>
    </div>
    <div style="background:#f8fafc;border:1px solid #cbd5e1;padding:12px;border-radius:8px;">
      <div style="font-size:10px;color:#475569;font-weight:700;text-transform:uppercase;">Net Objective</div>
      <div style="font-size:19px;font-weight:800;color:#334155;margin-top:2px;">-${abs(net_obj):.2f}k</div>
      <div style="font-size:11px;color:#64748b;margin-top:2px;">Includes penalty</div>
    </div>
    <div style="background:#fff;border:1px solid #e2e8f0;padding:12px;border-radius:8px;">
      <div style="font-size:10px;color:#64748b;font-weight:700;text-transform:uppercase;">Fill Rate</div>
      <div style="font-size:19px;font-weight:800;color:#2563eb;margin-top:2px;">{service_level:.1f}%</div>
      <div style="font-size:11px;color:#64748b;margin-top:2px;">15.0 / 15.45 u</div>
    </div>
  </div>

  <!-- Factory Operations & Warehouses Grid -->
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:15px;margin-bottom:20px;">
    <!-- Factory Utilization -->
    <div style="background:#fff;border:1px solid #e2e8f0;border-radius:10px;padding:15px;">
      <h3 style="margin:0 0 12px 0;font-size:13px;font-weight:700;color:#0f172a;text-transform:uppercase;">Factory Fleet Output & Capacities</h3>
      <div style="display:flex;flex-direction:column;gap:10px;font-size:12px;">
        <div style="background:#f8fafc;padding:10px;border-radius:6px;border-left:3px solid #2563eb;">
          <div style="display:flex;justify-content:space-between;font-weight:bold;">
            <span>Factory 1</span>
            <span style="color:#b91c1c;">5.0 / 5.0 u (100% Cap)</span>
          </div>
          <div style="color:#64748b;font-size:11px;margin-top:2px;">Product 1: 2.30 u | Product 2: 2.70 u | Op 1 Cap: 100% | Op 2 Cap: 96.3%</div>
        </div>
        <div style="background:#f8fafc;padding:10px;border-radius:6px;border-left:3px solid #d97706;">
          <div style="display:flex;justify-content:space-between;font-weight:bold;">
            <span>Factory 2</span>
            <span style="color:#b91c1c;">5.0 / 5.0 u (100% Cap)</span>
          </div>
          <div style="color:#64748b;font-size:11px;margin-top:2px;">Product 1: 2.00 u | Product 2: 3.00 u | Op 1 Cap: 100% | Op 2 Cap: 100%</div>
        </div>
        <div style="background:#f8fafc;padding:10px;border-radius:6px;border-left:3px solid #16a34a;">
          <div style="display:flex;justify-content:space-between;font-weight:bold;">
            <span>Factory 3 (Lowest Cost)</span>
            <span style="color:#b91c1c;">5.0 / 5.0 u (100% Cap)</span>
          </div>
          <div style="color:#64748b;font-size:11px;margin-top:2px;">Product 1: 2.00 u | Product 2: 3.00 u | Op 1 Cap: 100% | Op 2 Cap: 100%</div>
        </div>
      </div>
    </div>

    <!-- Warehouse & Logistics Hubs -->
    <div style="background:#fff;border:1px solid #e2e8f0;border-radius:10px;padding:15px;">
      <h3 style="margin:0 0 12px 0;font-size:13px;font-weight:700;color:#0f172a;text-transform:uppercase;">Warehouse Network Throughput</h3>
      <div style="display:grid;grid-template-columns:repeat(5, 1fr);gap:8px;text-align:center;font-size:11px;">
        <div style="background:#f1f5f9;padding:10px 4px;border-radius:6px;">
          <div style="font-weight:bold;color:#1e293b;">W1</div>
          <div style="font-size:14px;font-weight:800;color:#0284c7;margin:3px 0;">1.30u</div>
          <div style="color:#64748b;font-size:10px;">Road Truck</div>
        </div>
        <div style="background:#eff6ff;border:1px solid #bfdbfe;padding:10px 4px;border-radius:6px;">
          <div style="font-weight:bold;color:#1e3a8a;">W2</div>
          <div style="font-size:14px;font-weight:800;color:#2563eb;margin:3px 0;">3.70u</div>
          <div style="color:#2563eb;font-weight:bold;font-size:10px;">Rail Mega-Hub</div>
        </div>
        <div style="background:#f8fafc;border:1px dashed #cbd5e1;padding:10px 4px;border-radius:6px;opacity:0.6;">
          <div style="font-weight:bold;color:#94a3b8;">W3</div>
          <div style="font-size:14px;font-weight:800;color:#94a3b8;margin:3px 0;">0.00u</div>
          <div style="color:#e11d48;font-weight:bold;font-size:10px;">Idle</div>
        </div>
        <div style="background:#eff6ff;border:1px solid #bfdbfe;padding:10px 4px;border-radius:6px;">
          <div style="font-weight:bold;color:#1e3a8a;">W4</div>
          <div style="font-size:14px;font-weight:800;color:#2563eb;margin:3px 0;">3.70u</div>
          <div style="color:#2563eb;font-weight:bold;font-size:10px;">Road Mega-Hub</div>
        </div>
        <div style="background:#f1f5f9;padding:10px 4px;border-radius:6px;">
          <div style="font-weight:bold;color:#1e293b;">W5</div>
          <div style="font-size:14px;font-weight:800;color:#0284c7;margin:3px 0;">0.85u</div>
          <div style="color:#64748b;font-size:10px;">Road Truck</div>
        </div>
      </div>
      <div style="margin-top:12px;font-size:11px;color:#64748b;line-height:1.4;">
        <b>Direct Factory Bypass Corridors:</b> F1 &rarr; M2 (1.30u) and F3 &rarr; M7, M8 (4.15u) bypass warehouses entirely, saving intermediate freight fees.
      </div>
    </div>
  </div>

  <!-- Market Demand & Fulfillment Matrix -->
  <div style="background:#fff;border:1px solid #e2e8f0;border-radius:10px;padding:15px;">
    <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:12px;">
      <h3 style="margin:0;font-size:13px;font-weight:700;color:#0f172a;text-transform:uppercase;">Market Demand & Fulfillment Matrix</h3>
      <span style="font-size:11px;color:#15803d;font-weight:700;background:#dcfce7;padding:2px 8px;border-radius:999px;">8 of 9 Markets 100% Satisfied</span>
    </div>
    <div style="overflow-x:auto;">
      <table style="width:100%;border-collapse:collapse;font-size:12px;text-align:left;">
        <thead>
          <tr style="background:#f8fafc;border-bottom:2px solid #e2e8f0;color:#475569;font-size:11px;text-transform:uppercase;">
            <th style="padding:8px 12px;">Market</th>
            <th style="padding:8px 12px;text-align:center;">P1 (Dem / Shipped)</th>
            <th style="padding:8px 12px;text-align:center;">P2 (Dem / Shipped)</th>
            <th style="padding:8px 12px;text-align:center;">Total Demand</th>
            <th style="padding:8px 12px;text-align:center;">Shortage</th>
            <th style="padding:8px 12px;text-align:center;">Fill Rate</th>
            <th style="padding:8px 12px;text-align:right;">Gross Revenue</th>
            <th style="padding:8px 12px;text-align:center;">Channel</th>
          </tr>
        </thead>
        <tbody>
          {mkt_rows}
        </tbody>
      </table>
    </div>
  </div>

  <!-- Managerial Overtime Scenario Callout -->
  <div style="margin-top:18px;background:#f0fdf4;border:1px solid #86efac;padding:14px 18px;border-radius:10px;display:flex;justify-content:space-between;align-items:center;">
    <div>
      <div style="font-weight:700;color:#166534;font-size:13px;">Executive Strategy: Overtime / Debottlenecking Opportunity (+10% Capacity)</div>
      <div style="color:#15803d;font-size:12px;margin-top:2px;">
        Expanding factory Operation 1 limit from 5.0 to 5.5 units per factory completely clears the 0.45 unit shortage in Market 4 and <b>swings bottom-line earnings from -$17.43k to +$28.18k (+$45.61k net turnaround)</b>.
      </div>
    </div>
  </div>

</div>
"""

# Save standalone HTML file to workspace
with open("managerial_dashboard.html", "w", encoding="utf-8") as f:
    f.write(dashboard_html)

# Display inside notebook
display(HTML(dashboard_html))



<IPython.core.display.HTML object>


Market,P1 (Dem / Shipped),P2 (Dem / Shipped),Total Demand,Shortage,Fill Rate,Gross Revenue,Channel
Market 1,0.75 / 0.75,1.00 / 1.00,1.75,-,100.0%,$10.38k,Via Warehouse
Market 2,1.30 / 1.30,0.00 / 0.00,1.30,-,100.0%,$9.10k,Direct (F1)
Market 3,0.85 / 0.85,1.45 / 1.45,2.30,-,100.0%,$12.61k,Via Warehouse
Market 4,0.75 / 0.75,1.25 / 0.80,2.00,0.45,77.5%,$9.70k,Via Warehouse
Market 5,0.55 / 0.55,0.95 / 0.95,1.50,-,100.0%,$9.09k,Via Warehouse
Market 6,0.20 / 0.20,0.60 / 0.60,0.80,-,100.0%,$4.42k,Via Warehouse
Market 7,0.70 / 0.70,1.50 / 1.50,2.20,-,100.0%,$14.08k,Direct (F3)
Market 8,0.45 / 0.45,1.50 / 1.50,1.95,-,100.0%,$10.74k,Direct (F3)
Market 9,0.75 / 0.75,0.90 / 0.90,1.65,-,100.0%,$9.93k,Via Warehouse
